# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyanshu-Technologies/flyrank-ML-track/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


I chose Logistic Regression as the first learned model for this Lane 2 ranking task.

The goal is to rank content pages for human review, so I will use the model's predicted probability of the observed declining label as the ranking score.

Logistic Regression is a good first model because it is simple, reproducible, and interpretable. It gives us a strong comparison against the Week-4 rule-based baseline without adding unnecessary complexity.

The model will use observable page-level signals available before the decision point. I will not use `trend_direction`, `trend_pct`, or other label-derived fields as features.

The main evaluation metric is Precision@50 because the content team has limited review capacity and we care about how many useful review candidates appear near the top of the queue.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Reproducibility
RANDOM_STATE = 42

# Load starter dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

# Create the observed label
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Declining rate:", df["is_declining_label"].mean())

Rows: 30000
Columns: 44
Declining rate: 0.5420666666666667


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a client-grouped train/test split.

The split is grouped by `client_id`, so all pages belonging to a client stay entirely in either the training set or the test set.

This is more honest than randomly splitting individual pages because pages from the same client can share characteristics. A random page-level split could therefore make the test set too similar to the training data.

I will use 80% of the clients for training and 20% for testing, with a fixed random seed for reproducibility.

The Week-4 baseline will be evaluated on this exact same test set and with the same Precision@50 metric.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

# Features available before the decision point
feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

target = "is_declining_label"
group = "client_id"

# Grouped train/test split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df[feature_columns],
        df[target],
        groups=df[group]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTraining clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

# Verify that no client appears in both sets
overlap = set(train_df["client_id"]) & set(test_df["client_id"])

print("\nClient overlap:", len(overlap))

assert len(overlap) == 0

print("Grouped split check: PASSED")


Training rows: 23837
Test rows: 6163

Training clients: 25
Test clients: 7

Client overlap: 0
Grouped split check: PASSED


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will train Logistic Regression using the grouped training set and use its predicted probability of decline as the ranking score.

I will evaluate both the learned model and the Week-4 rule-based baseline on the same held-out test clients.

The main metric is Precision@50. This measures how many of the top 50 ranked pages are actually labeled as declining.

The model is useful only if it provides a better ranking than the transparent Week-4 baseline on the same evaluation slice.

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# 1. Prepare train/test data
X_train = train_df[feature_columns].copy()
y_train = train_df[target].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[target].copy()


# 2. Build Logistic Regression
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])


# 3. Train model
print("Training Logistic Regression...")

model.fit(X_train, y_train)

print("Model trained!")


# 4. Get probability scores
test_df["model_score"] = model.predict_proba(
    X_test
)[:, 1]



# 5. Precision@50 function
def precision_at_k(data, score_column, k=50):
    ranked = data.sort_values(
        score_column,
        ascending=False
    ).head(k)

    return ranked[target].mean()


# 6. Create W04 baseline score
#    on the SAME test set
test_df["baseline_score"] = (
    test_df["impressions_90d"]
    * (test_df["days_since_last_update"] >= 180).astype(int)
    * (test_df["impressions_90d"] >= 500).astype(int)
)


# 7. Evaluate both
model_precision = precision_at_k(
    test_df,
    "model_score",
    k=50
)

baseline_precision = precision_at_k(
    test_df,
    "baseline_score",
    k=50
)



# 8. Create comparison table
comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "Precision@50": [
        baseline_precision,
        model_precision
    ]
})

comparison["Precision@50"] = (
    comparison["Precision@50"].round(3)
)

display(comparison)


# 9. Show base rate
print(
    "\nTest-set declining base rate:",
    round(y_test.mean(), 3)
)

print(
    "\nModel improvement over baseline:",
    round(
        model_precision - baseline_precision,
        3
    )
)


Training Logistic Regression...
Model trained!


,method,Precision@50
0,Week-4 baseline,0.7
1,Logistic Regression,1.0



Test-set declining base rate: 0.511

Model improvement over baseline: 0.3


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I will inspect the highest-ranked model predictions and identify where the model is correct and where it is wrong.

A false positive is a page ranked highly by the model but not labeled as declining. These pages are important because they could consume review capacity without matching the observed decline signal.

A false negative is a declining page that receives a relatively low model score. These pages matter because the ranking could miss pages that deserve human attention.

I will also inspect the model coefficients to understand which observable features are associated with higher or lower predicted decline probability.

These relationships are predictive associations in this dataset, not causal effects.

In [4]:
# 1. Inspect top 20 predictions
top20_model = (
    test_df
    .sort_values("model_score", ascending=False)
    .head(20)
    .copy()
)

top20_model["prediction"] = (
    top20_model["model_score"] >= 0.5
).astype(int)

top20_model["error_type"] = np.where(
    (top20_model["prediction"] == 1) &
    (top20_model[target] == 0),
    "false_positive",
    np.where(
        (top20_model["prediction"] == 0) &
        (top20_model[target] == 1),
        "false_negative",
        "correct"
    )
)

print("Top 20 model-ranked pages:")
display(
    top20_model[
        [
            "content_id",
            "client_id",
            "model_score",
            target,
            "error_type",
            "impressions_90d",
            "days_since_last_update",
            "avg_position"
        ]
    ]
)



# 2. Count errors
false_positives = (
    (test_df["model_score"] >= 0.5) &
    (test_df[target] == 0)
).sum()

false_negatives = (
    (test_df["model_score"] < 0.5) &
    (test_df[target] == 1)
).sum()

print("\nFalse positives:", false_positives)
print("False negatives:", false_negatives)



# 3. Inspect Logistic Regression
#    feature coefficients
classifier = model.named_steps["classifier"]

coefficients = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": classifier.coef_[0]
})

coefficients["abs_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "abs_coefficient",
    ascending=False
)

print("\nMost influential features:")
display(
    coefficients[
        ["feature", "coefficient"]
    ].head(10)
)


# 4. Final interpretation
print("\nInterpretation:")

if model_precision > baseline_precision:
    print(
        "Logistic Regression beats the Week-4 baseline on Precision@50 "
        "on the same held-out test clients."
    )
elif model_precision < baseline_precision:
    print(
        "The Week-4 baseline beats Logistic Regression on Precision@50 "
        "on the same held-out test clients."
    )
else:
    print(
        "Logistic Regression and the Week-4 baseline have the same "
        "Precision@50 on the held-out test clients."
    )

print(
    "The model errors show where a learned ranking can disagree with "
    "the observed declining label. These errors should be reviewed "
    "before treating the model as useful decision support."
)


Top 20 model-ranked pages:


,content_id,client_id,model_score,is_declining_label,error_type,impressions_90d,days_since_last_update,avg_position
2515,content_2dc625dcdf98,client_4e07408562,1.0,1,correct,72671,25,8.5
14949,content_bf658584d14d,client_f369cb89fc,1.0,1,correct,31511,20,6.7
14638,content_bfeab0d95550,client_f369cb89fc,1.0,1,correct,41474,20,23.0
14226,content_65d9331f55fb,client_4e07408562,1.0,1,correct,65686,7,7.8
16174,content_1f37db922da1,client_4e07408562,1.0,1,correct,74642,25,6.4
26413,content_bdd7c88a58ed,client_4e07408562,1.0,1,correct,64718,14,3.1
2070,content_b138c0b35a91,client_4e07408562,1.0,1,correct,29545,7,5.0
15914,content_0919dd345d80,client_4e07408562,1.0,1,correct,119217,7,7.0
13842,content_c97ce8d47eef,client_f369cb89fc,1.0,1,correct,29836,20,8.3
13705,content_25f7bcf2a206,client_f369cb89fc,1.0,1,correct,21890,20,34.4



False positives: 515
False negatives: 993

Most influential features:


,feature,coefficient
15,impressions_last_30d,-35.021821
18,impressions_prev_30d,29.228892
6,impressions_90d,1.332100
17,sessions_last_30d,-0.798518
16,clicks_last_30d,-0.788385
19,clicks_prev_30d,0.779447
9,sessions_90d,0.751389
8,pageviews_90d,0.745031
10,users_90d,-0.724345
13,days_with_impressions,0.549279



Interpretation:
Logistic Regression beats the Week-4 baseline on Precision@50 on the same held-out test clients.
The model errors show where a learned ranking can disagree with the observed declining label. These errors should be reviewed before treating the model as useful decision support.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.